In [ ]:
# Interpretability — SHAP & LIME

This notebook applies two explainability methods to the trained U-Net segmentation model:

- **SHAP** (GradientExplainer): pixel-level gradient-based attributions showing which input regions drive the model's predictions.
- **LIME** (LimeImageExplainer): superpixel-based perturbation approach showing which image segments contribute most to the predicted tumor region.

Both methods help understand *why* the model segments a region as tumor.

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

sys.path.append(os.path.abspath(".."))
from dataset import BrainTumorDataset
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.encoder = nn.ModuleList()
        self.decoder = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)

        for f in features:
            self.encoder.append(DoubleConv(in_channels, f))
            in_channels = f

        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.decoder.append(nn.ConvTranspose2d(f * 2, f, 2, 2))
            self.decoder.append(DoubleConv(f * 2, f))

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoder:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skips = skips[::-1]

        for i in range(0, len(self.decoder), 2):
            x = self.decoder[i](x)
            skip = skips[i // 2]
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = self.decoder[i + 1](x)

        return torch.sigmoid(self.final(x))


model = UNet().to(device)
print("U-Net loaded.")


In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())

dataset = BrainTumorDataset(
    image_dir=os.path.join(BASE_DIR, "data/processed/images"),
    mask_dir=os.path.join(BASE_DIR, "data/processed/masks"),
)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
_, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

print("Sample batch — images:", images.shape, "masks:", masks.shape)


## SHAP — Gradient-Based Pixel Attribution

`shap.GradientExplainer` computes expected gradients with respect to a background dataset.  
For each pixel, the SHAP value indicates how much that pixel pushed the model's mean output (tumor probability) above or below the background average.

Positive values (red) = pixels that increase predicted tumor probability.  
Negative values (blue) = pixels that suppress it.

In [ ]:
class UNetScalarWrapper(nn.Module):
    def __init__(self, unet):
        super().__init__()
        self.unet = unet

    def forward(self, x):
        seg_map = self.unet(x)
        return seg_map.mean(dim=[1, 2, 3], keepdim=False).unsqueeze(1)


scalar_model = UNetScalarWrapper(model).to(device)
scalar_model.eval()

background = images[:4]
explainer = shap.GradientExplainer(scalar_model, background)

test_images = images[4:8]
shap_values = explainer.shap_values(test_images)

print("SHAP values shape:", np.array(shap_values).shape)


In [ ]:
n_samples = 4
fig, axes = plt.subplots(n_samples, 3, figsize=(12, n_samples * 4))

for i in range(n_samples):
    img_np = test_images[i].cpu().permute(1, 2, 0).numpy()
    mask_np = masks[4 + i].cpu().squeeze().numpy()

    shap_map = np.array(shap_values)[0, i]
    shap_agg = shap_map.sum(axis=0)

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("MRI Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

    vmax = np.abs(shap_agg).max()
    axes[i, 2].imshow(img_np)
    im = axes[i, 2].imshow(shap_agg, cmap="RdBu_r", alpha=0.6, vmin=-vmax, vmax=vmax)
    axes[i, 2].set_title("SHAP Attribution")
    axes[i, 2].axis("off")
    plt.colorbar(im, ax=axes[i, 2], fraction=0.046, pad=0.04)

plt.suptitle("SHAP Explanations — U-Net Brain Tumor Segmentation", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "reports/shap_explanation.png"), dpi=150, bbox_inches="tight")
plt.show()
print("SHAP plot saved to reports/shap_explanation.png")


## LIME — Superpixel Perturbation Analysis

`LimeImageExplainer` masks out groups of pixels (superpixels) and observes how the model's tumor probability score changes.  
Superpixels highlighted in green increase the predicted tumor score — i.e., they are the regions the model relies on most.

In [ ]:
def lime_predict(images_np):
    batch = torch.tensor(images_np, dtype=torch.float32).permute(0, 3, 1, 2).to(device)
    with torch.no_grad():
        preds = model(batch)
    tumor_score = preds.mean(dim=[1, 2, 3]).cpu().numpy()
    no_tumor_score = 1.0 - tumor_score
    return np.stack([no_tumor_score, tumor_score], axis=1)


explainer_lime = lime_image.LimeImageExplainer()

lime_explanations = []
for i in range(4):
    img_np = test_images[i].cpu().permute(1, 2, 0).numpy().astype(np.float64)
    explanation = explainer_lime.explain_instance(
        img_np,
        lime_predict,
        top_labels=1,
        hide_color=0,
        num_samples=200,
    )
    lime_explanations.append(explanation)
    print(f"LIME explanation {i+1}/4 done.")


In [ ]:
fig, axes = plt.subplots(n_samples, 3, figsize=(12, n_samples * 4))

for i in range(n_samples):
    img_np = test_images[i].cpu().permute(1, 2, 0).numpy()
    mask_np = masks[4 + i].cpu().squeeze().numpy()
    exp = lime_explanations[i]

    top_label = exp.top_labels[0]
    temp, lime_mask = exp.get_image_and_mask(
        top_label,
        positive_only=True,
        num_features=10,
        hide_rest=False,
    )

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("MRI Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(mark_boundaries(temp, lime_mask))
    axes[i, 2].set_title("LIME Important Regions")
    axes[i, 2].axis("off")

plt.suptitle("LIME Explanations — U-Net Brain Tumor Segmentation", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "reports/lime_explanation.png"), dpi=150, bbox_inches="tight")
plt.show()
print("LIME plot saved to reports/lime_explanation.png")


## SHAP vs LIME — Side-by-Side Comparison

Both methods are shown together for the same samples so their explanations can be directly compared.

In [ ]:
fig, axes = plt.subplots(n_samples, 4, figsize=(16, n_samples * 4))

for i in range(n_samples):
    img_np = test_images[i].cpu().permute(1, 2, 0).numpy()
    mask_np = masks[4 + i].cpu().squeeze().numpy()

    shap_map = np.array(shap_values)[0, i].sum(axis=0)
    vmax = np.abs(shap_map).max()

    exp = lime_explanations[i]
    top_label = exp.top_labels[0]
    temp, lime_mask = exp.get_image_and_mask(
        top_label, positive_only=True, num_features=10, hide_rest=False
    )

    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title("MRI Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].set_title("Ground Truth Mask")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(img_np)
    im = axes[i, 2].imshow(shap_map, cmap="RdBu_r", alpha=0.6, vmin=-vmax, vmax=vmax)
    axes[i, 2].set_title("SHAP Attribution")
    axes[i, 2].axis("off")
    plt.colorbar(im, ax=axes[i, 2], fraction=0.046, pad=0.04)

    axes[i, 3].imshow(mark_boundaries(temp, lime_mask))
    axes[i, 3].set_title("LIME Regions")
    axes[i, 3].axis("off")

plt.suptitle("SHAP vs LIME — U-Net Brain Tumor Segmentation", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "reports/shap_vs_lime.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Comparison plot saved to reports/shap_vs_lime.png")
